# Notebook 1 — Extract face images from FF++ and Celeb-DF videos

**Kaggle settings (right panel):**
- Add Input: `xdxd003/ff-c23` and `reubensuju/celeb-df-v2`
- Accelerator: **None** (CPU only, saves GPU hours)
- Internet: **On** (to download the small face-detector model)

**Output:** `/kaggle/working/faces/` with cropped 256x256 faces + `metadata.csv`

After it finishes: **Save Version → Save & Run All (Commit)**, then open the
version's **Output** tab → **New Dataset** → name it `deepfake-faces`.
All later notebooks use that dataset.

In [ ]:
import os, glob, urllib.request
from collections import Counter
from concurrent.futures import ProcessPoolExecutor

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

print("OpenCV version:", cv2.__version__)  # needs 4.8 or newer

# ---------------- settings ----------------
OUT = "/kaggle/working/faces"
FRAMES_FF_REAL = 8      # frames per real FF++ video
FRAMES_FF_FAKE = 2      # frames per fake FF++ video (4 fake types -> balanced with real)
FRAMES_CELEB = 8        # frames per Celeb-DF test video
IMG_SIZE = 256
MARGIN = 1.3            # crop a bit larger than the face box
FF_FAKE_TYPES = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]
CELEB_DIRS = {"Celeb-real", "Celeb-synthesis", "YouTube-real"}

MODEL_URL = ("https://github.com/opencv/opencv_zoo/raw/main/models/"
             "face_detection_yunet/face_detection_yunet_2023mar.onnx")
MODEL_PATH = "/kaggle/working/yunet.onnx"
if not os.path.exists(MODEL_PATH):
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print("Face detector ready")

## Step 1 — Find all videos and check folder names

In [ ]:
all_videos = sorted(glob.glob("/kaggle/input/**/*.mp4", recursive=True))
print("Total videos found:", len(all_videos))
print("Videos per folder:")
for k, v in Counter(os.path.basename(os.path.dirname(p)) for p in all_videos).most_common():
    print(f"  {k:30s} {v}")

def path_parts(p):
    return set(p.split(os.sep))

celeb_videos = [p for p in all_videos if CELEB_DIRS & path_parts(p)]
celeb_set = set(celeb_videos)
ff_videos = [p for p in all_videos if p not in celeb_set]
print("\nFF++ videos:", len(ff_videos), "| Celeb-DF videos:", len(celeb_videos))

## Step 2 — Build the list of jobs
FF++ split (same sizes as the official 720/140/140 split):
video id 000-719 = train, 720-859 = val, 860-999 = test.
Celeb-DF: only the 518 official test videos.

In [ ]:
def ff_label(p):
    ps = path_parts(p)
    if "original" in ps or "original_sequences" in ps:
        return 0, "original"
    for t in FF_FAKE_TYPES:
        if t in ps:
            return 1, t
    return None, None   # skips FaceShifter / DeepFakeDetection etc.

def ff_split(p):
    vid = int(os.path.basename(p).split(".")[0].split("_")[0])
    return "train" if vid < 720 else ("val" if vid < 860 else "test")

jobs = []
for p in ff_videos:
    label, method = ff_label(p)
    if label is None:
        continue
    split = ff_split(p)
    name = os.path.basename(p).split(".")[0]
    cls = "real" if label == 0 else "fake"
    jobs.append(dict(path=p, dataset="ffpp", split=split, label=label, method=method,
                     video=f"{method}_{name}",
                     out_dir=f"{OUT}/ffpp/{split}/{cls}",
                     n=FRAMES_FF_REAL if label == 0 else FRAMES_FF_FAKE))

# Celeb-DF official test list: lines like "1 YouTube-real/00170.mp4" (1 = real, 0 = fake)
list_files = glob.glob("/kaggle/input/**/List_of_testing_videos.txt", recursive=True)
assert list_files, "List_of_testing_videos.txt not found - check the Celeb-DF input"
celeb_lookup = {os.path.join(os.path.basename(os.path.dirname(p)), os.path.basename(p)): p
                for p in celeb_videos}
missing = 0
with open(list_files[0]) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        flag, rel = line.split()
        if rel not in celeb_lookup:
            missing += 1
            continue
        label = 0 if flag == "1" else 1
        cls = "real" if label == 0 else "fake"
        folder, fname = rel.split("/")
        jobs.append(dict(path=celeb_lookup[rel], dataset="celebdf", split="test", label=label,
                         method=folder, video=f"{folder}_{fname.split('.')[0]}",
                         out_dir=f"{OUT}/celebdf/test/{cls}", n=FRAMES_CELEB))
print("Celeb-DF test videos missing from input:", missing)

jdf = pd.DataFrame(jobs)
print(jdf.groupby(["dataset", "split", "method"]).size())

## Step 3 — Face detection + cropping (runs on all CPU cores)

In [ ]:
_det = None
def get_detector():
    global _det
    if _det is None:
        cv2.setNumThreads(1)
        _det = cv2.FaceDetectorYN.create(MODEL_PATH, "", (320, 320), 0.6, 0.3, 5000)
    return _det

def crop_face(frame):
    h, w = frame.shape[:2]
    s = 640 / max(h, w) if max(h, w) > 640 else 1.0     # detect on a smaller copy (faster)
    small = cv2.resize(frame, (int(w * s), int(h * s))) if s < 1 else frame
    det = get_detector()
    det.setInputSize((small.shape[1], small.shape[0]))
    _, faces = det.detect(small)
    if faces is None or len(faces) == 0:
        return None
    x, y, bw, bh = max(faces, key=lambda f: f[2] * f[3])[:4] / s   # largest face
    cx, cy, side = x + bw / 2, y + bh / 2, max(bw, bh) * MARGIN
    x1, y1 = int(max(0, cx - side / 2)), int(max(0, cy - side / 2))
    x2, y2 = int(min(w, cx + side / 2)), int(min(h, cy + side / 2))
    face = frame[y1:y2, x1:x2]
    if face.size == 0:
        return None
    return cv2.resize(face, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

def process_video(job):
    os.makedirs(job["out_dir"], exist_ok=True)
    cap = cv2.VideoCapture(job["path"])
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    rows = []
    if total > 0:
        idxs = np.linspace(0, total - 1, job["n"] + 2, dtype=int)[1:-1]  # skip first/last frame
        for i in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
            ok, frame = cap.read()
            if not ok:
                continue
            face = crop_face(frame)
            if face is None:
                continue
            fname = f"{job['video']}_f{i:04d}.png"
            cv2.imwrite(os.path.join(job["out_dir"], fname), face)
            rows.append(dict(path=os.path.relpath(os.path.join(job["out_dir"], fname), OUT),
                             label=job["label"], dataset=job["dataset"], split=job["split"],
                             method=job["method"], video=job["video"]))
    cap.release()
    return rows

all_rows = []
with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
    for rows in tqdm(ex.map(process_video, jobs, chunksize=4), total=len(jobs)):
        all_rows.extend(rows)

meta = pd.DataFrame(all_rows)
meta.to_csv(f"{OUT}/metadata.csv", index=False)
print("Saved faces:", len(meta))

## Step 4 — Check the result

In [ ]:
print(meta.groupby(["dataset", "split", "label"]).size().unstack(fill_value=0)
          .rename(columns={0: "real", 1: "fake"}))

import matplotlib.pyplot as plt
sample = meta.groupby(["dataset", "label"]).sample(4, random_state=0)
fig, axes = plt.subplots(len(sample) // 8 + (len(sample) % 8 > 0), 8, figsize=(16, 4.5))
for ax in axes.ravel():
    ax.axis("off")
for ax, (_, r) in zip(axes.ravel(), sample.iterrows()):
    img = cv2.imread(os.path.join(OUT, r["path"]))[:, :, ::-1]
    ax.imshow(img)
    ax.set_title(f"{r['dataset']} {'fake' if r['label'] else 'real'}", fontsize=8)
plt.tight_layout()
plt.show()

os.remove(MODEL_PATH)  # keep the output dataset clean